# E008 — Drift, staleness and exposure re-balancing

**Hypotheses:** H9 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** What drifts across the ten days, what does a stale model cost, and can an adaptation layer spread exposure without losing CTR?

**Method:** Daily PSI against the training window, churn statistics, a staleness backtest of models frozen at different days, a causal per-genre hourly recalibration, and a replay of a cohort-exposure penalty with doubly robust CTR estimates.

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
import matplotlib.pyplot as plt
import polars as pl

from charade.analysis import adaptation, drift
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
tables = drift.run(ctx.settings, ctx.data_dir, ctx.reports / "drift")
tables["Daily mix and churn"]

day,impressions,ctr,first_seen_character_share,top100_jaccard_prev_day,character_hhi,new_creative_share
str,i64,f64,f64,f64,f64,f64
"""2014-10-21""",111065,0.179183,1.0,NaN,0.000656,1.0
"""2014-10-22""",143836,0.166627,0.010846,0.754386,0.000618,0.34433
"""2014-10-23""",104452,0.191638,0.005256,0.652893,0.000602,0.22282
"""2014-10-24""",89378,0.184329,0.005012,0.538462,0.000586,0.132214
"""2014-10-25""",90218,0.192545,0.008402,0.481481,0.000585,0.025749
"""2014-10-26""",103948,0.194972,0.005281,0.538462,0.000583,0.026369
"""2014-10-27""",86788,0.195465,0.003365,0.5625,0.000594,0.133786
"""2014-10-28""",142909,0.165357,0.007081,0.550388,0.000608,0.342407
"""2014-10-29""",104450,0.172705,0.003973,0.680672,0.000629,0.235807


In [3]:
stale = tables["Staleness: frozen models scored on later days (1 seed each)"]
fig, ax = plt.subplots(figsize=(6, 4))
for (day,), part in stale.group_by("scored_day", maintain_order=True):
    ax.plot(part["age_days"], part["ne"], "o-", label=str(day))
ax.set(xlabel="model age (days since last training day)", ylabel="NE", title="Staleness")
ax.legend(title="scored day", fontsize=7)
plt.show()

/tmp/ipykernel_1185021/550794735.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
adapt = adaptation.run(ctx.settings, ctx.data_dir, ctx.reports / "drift" / "adaptation.md")
adapt.with_columns(pl.col(pl.Float64).round(4))

lambda,dr_ctr,dr_delta_vs_greedy,ci_low,ci_high,cohort_dominance,cohort_hhi,repeat_exposure
f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.1836,0.0,0.0,0.0,0.0768,0.0271,0.1047
1.0,0.1823,-0.0013,-0.0085,0.0054,0.0621,0.022,0.0974
2.0,0.1832,-0.0004,-0.008,0.0068,0.0579,0.0205,0.0957
4.0,0.1826,-0.001,-0.0098,0.0074,0.0538,0.0187,0.0929
8.0,0.1812,-0.0024,-0.0114,0.0061,0.0471,0.0164,0.0903
16.0,0.1815,-0.0021,-0.0116,0.0071,0.041,0.0143,0.0877


## Reading
- The CTR level moves for every genre together; character preferences are stable.
- NE worsens with model age on every scored day, hence daily retraining.
- Higher lambda trades concentration for estimated CTR; at lambda = 4 the CTR change is within noise.

## Conclusion
Ads rotate fast (13-37 % new creatives a day) and a frozen model loses ~0.003 NE per day, so retrain daily; online recalibration adds nothing; an exposure penalty (lambda = 4) cuts cohort concentration 30 % with no detectable CTR loss.

**Decision:** Daily retraining with gated promotion; no online recalibration; exposure penalty recommended at lambda = 4 pending an online A/B test.